In [1]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize
from statsmodels.stats.power import NormalIndPower

np.random.seed(42)

In [2]:
# Load the dataset
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [3]:
# Check basic info and the first few rows
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Data Preparation

In [4]:
# Check distribution for Task 1
print("Payment Method counts (Task 1):")
print(df['PaymentMethod'].value_counts())

# Check distribution for Task 2
print("\nContract vs Churn contingency table (Task 2):")
print(pd.crosstab(df['Churn'], df['Contract']))

Payment Method counts (Task 1):
PaymentMethod
Electronic check             2365
Mailed check                 1612
Bank transfer (automatic)    1544
Credit card (automatic)      1522
Name: count, dtype: int64

Contract vs Churn contingency table (Task 2):
Contract  Month-to-month  One year  Two year
Churn                                       
No                  2220      1307      1647
Yes                 1655       166        48


### Task 1: Chi-Square Goodness-of-Fit Test

**Scenario:** A mobile-game studio expects that players choose among four character classes in equal proportions (25 % each). After a recent update, they collected the following counts from 400 new players:

| Class | Warrior | Mage | Rogue | Healer |
|-------|---------|------|-------|--------|
| Count | 120     | 85   | 110   | 85     |

1. Store the observed counts in a NumPy array.
2. Compute the expected counts under the null hypothesis of equal preference.
3. Run `st.chisquare(observed, f_exp=expected)`.
4. Report the χ² statistic, degrees of freedom, and p-value.
5. At α = 0.05, state whether you reject or fail to reject H₀.
6. In a Markdown cell, answer: "Which class (if any) appears over- or under-represented, and by how much?"

In [5]:
# 1. Store the observed counts in a NumPy array
observed_counts = df['PaymentMethod'].value_counts().values
categories = df['PaymentMethod'].value_counts().index

# 2. Compute the expected counts under the null hypothesis of equal preference
total_observations = len(df)
num_categories = len(observed_counts)
expected_counts = np.full(num_categories, total_observations / num_categories)

# 3. Run the chi-square goodness-of-fit test
chi2_stat, p_value = st.chisquare(observed_counts, f_exp=expected_counts)

# 4. Calculate degrees of freedom
df_degrees = num_categories - 1

# Report the results
print(f"Chi-Square Statistic: {chi2_stat:.4f}")
print(f"Degrees of Freedom: {df_degrees}")
print(f"P-value: {p_value:.4e}")

Chi-Square Statistic: 278.9872
Degrees of Freedom: 3
P-value: 3.5073e-60


* **Result Reporting:**
    * $\chi^2$ statistic: $278.9872$
    * Degrees of Freedom ($df$): $3$
    * p-value: $3.5073 \times 10^{-60}$
* **Decision:** At $\alpha = 0.05$, we **reject the null hypothesis ($H_0$)** because the p-value is significantly smaller than $0.05$. This indicates that the distribution of payment methods is not uniform; customers do have specific preferences.
* **Representation Analysis:** The **Electronic check** method is significantly **over-represented**. With an expected count of $1760.75$, its observed count of $2365$ is roughly $604$ higher than expected.
    * **Credit card (automatic)** is the most **under-represented**, having approximately $239$ fewer occurrences than expected ($1522$ observed vs $1760.75$ expected).

### Task 2: Chi-Square Test of Independence

**Scenario:** A marketing team wants to know whether subscription tier (Free, Basic, Premium) is associated with churn status (Churned, Retained). They collected data from 600 customers:

| | Free | Basic | Premium |
|---------|------|-------|---------|
| Churned | 90   | 60    | 30      |
| Retained| 110  | 140   | 170     |

1. Create the contingency table as a 2 × 3 NumPy array or pandas DataFrame.
2. Run `st.chi2_contingency(table)` and unpack χ², p-value, degrees of freedom, and expected frequencies.
3. Display the expected-frequency table.
4. Report the χ² statistic, df, and p-value.
5. State your decision and a plain-language interpretation (e.g. "There is / is not evidence of an association between subscription tier and churn.").

In [6]:
# 1. Create the contingency table (Churn vs Contract)
contingency_table = pd.crosstab(df['Churn'], df['Contract'])

# 2. Run st.chi2_contingency and unpack results
chi2_stat, p_val, dof, expected_freq = st.chi2_contingency(contingency_table)

# 3. Display the expected-frequency table as a DataFrame for readability
expected_df = pd.DataFrame(
    expected_freq, 
    index=contingency_table.index, 
    columns=contingency_table.columns
)

# 4. Report the results
print("--- Chi-Square Test of Independence ---")
print("\nObserved Contingency Table:")
print(contingency_table)
print("\nExpected Frequency Table:")
print(expected_df)
print(f"\nChi-Square Statistic: {chi2_stat:.4f}")
print(f"Degrees of Freedom: {dof}")
print(f"P-value: {p_val:.4e}")

--- Chi-Square Test of Independence ---

Observed Contingency Table:
Contract  Month-to-month  One year  Two year
Churn                                       
No                  2220      1307      1647
Yes                 1655       166        48

Expected Frequency Table:
Contract  Month-to-month    One year     Two year
Churn                                            
No           2846.691751  1082.11018  1245.198069
Yes          1028.308249   390.88982   449.801931

Chi-Square Statistic: 1184.5966
Degrees of Freedom: 2
P-value: 5.8630e-258


* **Results Reporting:**
    * $\chi^2$ statistic: $1184.5966$
    * Degrees of Freedom ($df$): $2$
    * p-value: $5.8630 \times 10^{-258}$
* **Decision:** At $\alpha = 0.05$, we **reject the null hypothesis ($H_0$)**.
* **Interpretation:** There is **overwhelming evidence of a significant association** between a customer's subscription tier (Contract) and whether or not they churn. The variables are not independent.

### Task 3: Compute Cramér's V

Using the χ² statistic and the contingency table from Task 2:

1. Write a function:

```python
def cramers_v(chi2, n, min_dim):
    """
    Returns Cramér's V.

    Parameters
    ----------
    chi2 : float
        Chi-square statistic.
    n : int
        Total number of observations.
    min_dim : int
        min(rows, cols) - 1 of the contingency table.
    """
```

2. Compute Cramér's V for the Task 2 result.
3. Classify the effect size using Cohen's benchmarks for df* = min(rows, cols) − 1:
   - Small ≈ 0.10, Medium ≈ 0.30, Large ≈ 0.50 (for 2 × 3 tables, df* = 1)
4. In a Markdown cell, answer: "Is the relationship between tier and churn weak, moderate, or strong? What does this mean for the marketing team?"

In [7]:
def cramers_v(chi2, n, min_dim):
    """
    Returns Cramér's V.

    Parameters
    ----------
    chi2 : float
        Chi-square statistic.
    n : int
        Total number of observations.
    min_dim : int
        min(rows, cols) - 1 of the contingency table.
    """
    return np.sqrt(chi2 / (n * min_dim))

# 2. Compute Cramér's V for the Task 2 result
n_total = contingency_table.values.sum()
# For a 2x3 table: min(2, 3) - 1 = 1
min_dimension = min(contingency_table.shape) - 1

v_result = cramers_v(chi2_stat, n_total, min_dimension)

print(f"Cramér's V: {v_result:.4f}")

Cramér's V: 0.4101


* **Result:** The calculated Cramér's V is **$0.4101$**.
* **Classification:** According to Cohen's benchmarks for $df^* = 1$ (where Small $\approx 0.10$, Medium $\approx 0.30$, and Large $\approx 0.50$), this value represents a **Moderate to Strong** effect.
* **Analysis for Marketing:**
    * **Strength:** The relationship between contract tier and churn is **moderate to strong**.
    * **Meaning:** This means that the contract type is a very meaningful predictor of customer retention. The association isn't just a statistical fluke; it is a substantial factor in the business.
    * **Recommendation:** Since month-to-month contracts are so strongly tied to churn, the marketing team should prioritize moving customers to "One year" or "Two year" contracts, as these tiers are strongly associated with higher retention.